<a href="https://colab.research.google.com/github/keyvanmahmoudi/Deep_Learning_Course/blob/main/solutions/Session16_gans_and_diffusion_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

# پاسخ تمرین‌های جلسهٔ ۱۶: مدل‌های مولد: GAN، Diffusion و پروژه پایانی

این نوت‌بوک پاسخ پیشنهادی تمرین‌های انتهای [نوت‌بوک جلسه](../sessions/Session16_gans_and_diffusion.ipynb) است. قبل از دیدن پاسخ‌ها، خودتان تمرین‌ها را حل کنید؛ معمولاً بیش از یک راه‌حل درست وجود دارد.

> **نکتهٔ اجرا:** بعضی پاسخ‌ها مدل‌های بزرگ‌تری آموزش می‌دهند؛ در Colab از منوی `Runtime → Change runtime type` گزینهٔ **T4 GPU** را انتخاب کنید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### آماده‌سازی

</div>

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from sklearn.datasets import make_moons

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_set = datasets.FashionMNIST('data', train=True, download=True, transform=tf)
CLASSES = train_set.classes
loader = DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2, drop_last=True)
Z_DIM, N_CLASSES = 64, 10
GAN_EPOCHS = 15 if device.type == 'cuda' else 3


class Generator(nn.Module):
    def __init__(self, z_dim=Z_DIM, n_classes=0):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, n_classes) if n_classes else None
        self.net = nn.Sequential(
            nn.Linear(z_dim + n_classes, 128 * 7 * 7), nn.BatchNorm1d(128 * 7 * 7), nn.ReLU(),
            nn.Unflatten(1, (128, 7, 7)),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.ReLU(),
            nn.ConvTranspose2d(64, 1, 4, stride=2, padding=1), nn.Tanh())

    def forward(self, z, labels=None):
        if self.label_emb is not None:
            z = torch.cat([z, self.label_emb(labels)], dim=1)          # condition on the class
        return self.net(z)


class Discriminator(nn.Module):
    def __init__(self, n_classes=0):
        super().__init__()
        self.label_emb = nn.Embedding(n_classes, 28 * 28) if n_classes else None
        in_ch = 2 if n_classes else 1                                    # the label becomes an extra image channel
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 4, stride=2, padding=1), nn.LeakyReLU(0.2),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2),
            nn.Flatten(), nn.Linear(128 * 7 * 7, 1))

    def forward(self, x, labels=None):
        if self.label_emb is not None:
            x = torch.cat([x, self.label_emb(labels).view(-1, 1, 28, 28)], dim=1)
        return self.net(x)


def train_gan(G, D, epochs=GAN_EPOCHS, lr_G=2e-4, lr_D=2e-4, conditional=False):
    G, D = G.to(device), D.to(device)
    opt_G = torch.optim.Adam(G.parameters(), lr=lr_G, betas=(0.5, 0.999))
    opt_D = torch.optim.Adam(D.parameters(), lr=lr_D, betas=(0.5, 0.999))
    bce = nn.BCEWithLogitsLoss()
    history = {'D(real)': [], 'D(fake)': []}
    for epoch in range(1, epochs + 1):
        G.train(); D.train()
        d_real_sum = d_fake_sum = n = 0
        for real, y in loader:
            real, y = real.to(device), y.to(device)
            b = real.size(0)
            labels = y if conditional else None
            ones, zeros = torch.ones(b, 1, device=device), torch.zeros(b, 1, device=device)
            fake = G(torch.randn(b, Z_DIM, device=device), labels)
            d_real, d_fake = D(real, labels), D(fake.detach(), labels)
            loss_D = bce(d_real, ones) + bce(d_fake, zeros)
            opt_D.zero_grad(); loss_D.backward(); opt_D.step()
            loss_G = bce(D(fake, labels), ones)
            opt_G.zero_grad(); loss_G.backward(); opt_G.step()
            d_real_sum += torch.sigmoid(d_real).sum().item(); d_fake_sum += torch.sigmoid(d_fake).sum().item(); n += b
        history['D(real)'].append(d_real_sum / n); history['D(fake)'].append(d_fake_sum / n)
        print(f"epoch {epoch:2d} | mean D(real) {history['D(real)'][-1]:.2f} | mean D(fake) {history['D(fake)'][-1]:.2f}")
    return history


def show(images, nrow, title):
    images = (images.clamp(-1, 1) * 0.5 + 0.5).cpu()
    ncol = math.ceil(len(images) / nrow)
    fig, axes = plt.subplots(nrow, ncol, figsize=(1.2 * ncol, 1.25 * nrow))
    for ax in axes.flat:
        ax.axis('off')
    for ax, img in zip(axes.flat, images):
        ax.imshow(img[0], cmap='gray')
    plt.suptitle(title); plt.tight_layout(); plt.show()


# ----- 2-D DDPM of the session notebook, generalised to any number of steps -----
def make_spiral(n=10_000, noise=0.03):
    t = np.sqrt(np.random.rand(n)) * 3 * np.pi
    x = np.stack([t * np.cos(t), t * np.sin(t)], 1) / (3 * np.pi)
    return torch.tensor(x + noise * np.random.randn(n, 2), dtype=torch.float32)


def make_schedule(T):
    """Linear betas scaled so that the total amount of noise matches the 200-step schedule of the session."""
    betas = torch.linspace(1e-4 * 200 / T, 0.05 * 200 / T, T).clamp(max=0.999)
    alphas = 1 - betas
    return betas, alphas, torch.cumprod(alphas, dim=0)


def time_embedding(t, dim=32):
    half = dim // 2
    freqs = torch.exp(-math.log(1000) * torch.arange(half) / half)
    angles = t.float().unsqueeze(1) * freqs.unsqueeze(0)
    return torch.cat([angles.sin(), angles.cos()], dim=1)


class NoisePredictor(nn.Module):
    def __init__(self, hidden=128, t_dim=32):
        super().__init__()
        self.t_dim = t_dim
        self.net = nn.Sequential(nn.Linear(2 + t_dim, hidden), nn.SiLU(), nn.Linear(hidden, hidden), nn.SiLU(),
                                 nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 2))

    def forward(self, x, t):
        return self.net(torch.cat([x, time_embedding(t, self.t_dim)], dim=1))


def train_ddpm(data, T, steps=6000, seed=0):
    torch.manual_seed(seed)
    betas, alphas, alpha_bar = make_schedule(T)
    model = NoisePredictor()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for _ in range(steps):
        x0 = data[torch.randint(len(data), (256,))]
        t = torch.randint(0, T, (256,))
        eps = torch.randn_like(x0)
        ab = alpha_bar[t].unsqueeze(1)
        loss = F.mse_loss(model(ab.sqrt() * x0 + (1 - ab).sqrt() * eps, t), eps)
        opt.zero_grad(); loss.backward(); opt.step()
    return model, (betas, alphas, alpha_bar)


@torch.no_grad()
def ddpm_sample(model, schedule, n=2000, seed=1):
    torch.manual_seed(seed)
    betas, alphas, alpha_bar = schedule
    x = torch.randn(n, 2)
    for t in reversed(range(len(betas))):
        eps = model(x, torch.full((n,), t))
        x = (x - betas[t] / (1 - alpha_bar[t]).sqrt() * eps) / alphas[t].sqrt()
        if t > 0:
            x = x + betas[t].sqrt() * torch.randn_like(x)
    return x


def chamfer(a, b):
    """Average distance from every point to the nearest point of the other set (lower = closer)."""
    d = torch.cdist(a, b)
    return (d.min(1).values.mean() + d.min(0).values.mean()).item() / 2


np.random.seed(0)
spiral = make_spiral()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۱

در GAN، نرخ یادگیری Discriminator را ۵ برابر کنید. چه اتفاقی برای $D(\text{fake})$ و کیفیت نمونه‌ها می‌افتد؟

</div>

In [ ]:
histories = {}
for name, lr_D in [('balanced (lr_D = 2e-4)', 2e-4), ('strong D (lr_D = 1e-3)', 1e-3)]:
    torch.manual_seed(0)
    G, D = Generator(), Discriminator()
    print(name)
    histories[name] = train_gan(G, D, lr_D=lr_D)
    G.eval()
    with torch.no_grad():
        show(G(torch.randn(16, Z_DIM, device=device)), 2, f'samples: {name}')

for name, h in histories.items():
    plt.plot(h['D(fake)'], 'o-', label=f'D(fake), {name}')
plt.axhline(0.5, ls='--', color='gray'); plt.xlabel('epoch'); plt.legend(); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

با نرخ یادگیری ۵ برابر، Discriminator سریع‌تر از Generator یاد می‌گیرد و $D(\text{fake})$ به سمت صفر می‌رود: تقریباً همهٔ نمونه‌های جعلی را با اطمینان تشخیص می‌دهد. نتیجه به میزان عدم تعادل بستگی دارد:

- در حالت شدید، گرادیانی که به Generator می‌رسد ضعیف یا بی‌جهت می‌شود و کیفیت نمونه‌ها پیشرفت نمی‌کند یا حتی بدتر می‌شود. نمونه‌ها نویزی و پر از الگوهای تکراری‌اند یا تنوعشان کم می‌شود (**Mode Collapse**).
- در حالت ملایم، به لطف تابع هزینهٔ Non-saturating، آموزش ادامه پیدا می‌کند ولی نوسان بیشتری دارد.

تعادل بین دو شبکه، و نه کمینه شدن یک Loss، معیار آموزش سالم GAN است. به همین دلیل نمودار $D(\text{real})$ و $D(\text{fake})$ را پایش می‌کنیم.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۲

یک **Conditional GAN** بسازید: برچسب کلاس را (با یک `nn.Embedding`) به ورودی هر دو شبکه اضافه کنید تا بتوانید تولید یک کلاس خاص (مثلاً فقط کیف) را درخواست کنید.

</div>

In [ ]:
torch.manual_seed(0)
cG, cD = Generator(n_classes=N_CLASSES), Discriminator(n_classes=N_CLASSES)
train_gan(cG, cD, conditional=True)

cG.eval()
with torch.no_grad():
    z = torch.randn(8, Z_DIM, device=device).repeat(N_CLASSES, 1)           # the same 8 noise vectors for every class
    labels = torch.arange(N_CLASSES, device=device).repeat_interleave(8)
    grid = cG(z, labels)
show(grid, N_CLASSES, 'conditional GAN: one row per class (' + ', '.join(CLASSES) + ')')

with torch.no_grad():
    bags = cG(torch.randn(16, Z_DIM, device=device), torch.full((16,), CLASSES.index('Bag'), device=device))
show(bags, 2, 'only bags, please')

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

برچسب به دو شکل به شبکه‌ها داده شد:

- در **Generator**، یک بردار Embedding کلاس کنار نویز $\mathbf{z}$ قرار می‌گیرد.
- در **Discriminator**، Embedding برچسب به یک «کانال تصویری» ۲۸×۲۸ تبدیل و کنار تصویر گذاشته می‌شود. حالا $D$ نه فقط «واقعی بودن» بلکه «واقعی بودن **برای این کلاس**» را می‌سنجد. پس Generator نمی‌تواند با تولید یک کفش واقعی وقتی کیف خواسته شده $D$ را فریب دهد.

در شبکهٔ نمونه‌ها، هر سطر یک کلاس است و هر ستون یک بردار نویز ثابت. معمولاً در هر ستون ویژگی‌هایی مثل ضخامت یا روشنایی مشابه است: **نویز «سبک» را کنترل می‌کند و برچسب «محتوا» را.** این ایده (Mirza & Osindero, 2014) پایهٔ مدل‌های شرطی امروزی است. در Stable Diffusion شرط به‌جای برچسب، یک جملهٔ متنی است.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۳

در DDPM دوبعدی، تعداد گام‌ها را ۲۰ و ۱۰۰۰ بگیرید (و `betas` را متناسب تنظیم کنید). کیفیت نمونه‌ها چه تغییری می‌کند؟

</div>

In [ ]:
samples = {}
for T in (20, 200, 1000):
    model, schedule = train_ddpm(spiral, T)
    samples[T] = ddpm_sample(model, schedule)
    print(f'T = {T:4d}: final alpha_bar {schedule[2][-1]:.4f} | chamfer distance to the data {chamfer(samples[T], spiral[:2000]):.4f}')

fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for ax, (T, x) in zip(axes, samples.items()):
    ax.scatter(x[:, 0], x[:, 1], s=2, color='C1'); ax.set_title(f'{T} steps')
axes[-1].scatter(spiral[:2000, 0], spiral[:2000, 1], s=2); axes[-1].set_title('real data')
for ax in axes:
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
plt.tight_layout(); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

`make_schedule` بتاها را طوری مقیاس می‌کند که **مقدار کل نویز** (و در نتیجه $\bar{\alpha}_T$) برای هر $T$ تقریباً همان مقدار جلسه باشد. اگر برای $T = 20$ همان بتاهای کوچک را نگه داریم، دادهٔ گام آخر هنوز نویز خالص نیست و فرض شروع از $\mathcal{N}(0, I)$ غلط می‌شود.

- با **۲۰ گام** هر گام معکوس باید مقدار زیادی نویز را یکجا حذف کند. فرض گاوسی بودن هر گام کوچک دیگر دقیق نیست و نمونه‌ها **پخش‌تر و تارتر** از مارپیچ واقعی‌اند.
- با **۱۰۰۰ گام** هر گام بسیار کوچک است و نمونه‌ها معمولاً دقیق‌تر روی مارپیچ می‌نشینند، اما تولید ۵۰ برابر کندتر از ۲۰ گام است.

این مصالحهٔ کیفیت و سرعت دلیل اصلی پیدایش روش‌هایی مثل **DDIM** است که با همان مدل آموزش‌دیده، با چند ده گام نمونه‌برداری می‌کنند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۴

DDPM را روی یک داده دوبعدی دیگر (مثلاً `make_moons` یا شکل یک حرف) آموزش دهید.

</div>

In [ ]:
moons = torch.tensor(make_moons(10_000, noise=0.05, random_state=0)[0], dtype=torch.float32)
moons = (moons - moons.mean(0)) / moons.std(0)                 # standardise, like the spiral scale


def letter_points(letter='K', n=10_000, noise=0.02, seed=0):
    """Sample points uniformly along the strokes of a block letter."""
    strokes = {'K': [((-1, -1.5), (-1, 1.5)), ((-1, 0), (1, 1.5)), ((-0.6, 0.3), (1, -1.5))]}[letter]
    rng = np.random.default_rng(seed)
    lengths = np.array([np.hypot(b[0] - a[0], b[1] - a[1]) for a, b in strokes])
    which = rng.choice(len(strokes), n, p=lengths / lengths.sum())
    u = rng.random(n)
    pts = np.array([np.array(strokes[w][0]) + u_i * (np.array(strokes[w][1]) - np.array(strokes[w][0]))
                    for w, u_i in zip(which, u)])
    return torch.tensor(pts + noise * rng.standard_normal(pts.shape), dtype=torch.float32)


letter = letter_points('K')
fig, axes = plt.subplots(1, 4, figsize=(17, 4))
for col, (name, data) in enumerate([('moons', moons), ('letter K', letter)]):
    model, schedule = train_ddpm(data, 200)
    gen = ddpm_sample(model, schedule)
    axes[2 * col].scatter(data[:2000, 0], data[:2000, 1], s=2); axes[2 * col].set_title(f'{name}: real')
    axes[2 * col + 1].scatter(gen[:, 0], gen[:, 1], s=2, color='C1'); axes[2 * col + 1].set_title(f'{name}: DDPM samples')
for ax in axes:
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

بدون هیچ تغییری در مدل یا الگوریتم، فقط با عوض کردن داده، DDPM توزیع جدید را یاد می‌گیرد. این همان ویژگی کلیدی مدل‌های مولد است: الگوریتم **عمومی** است و دانش مسئله فقط از داده می‌آید. داده را استاندارد کردیم تا مقیاسش با نویز $\mathcal{N}(0, I)$ هم‌خوان باشد؛ این کار برای Diffusion مهم است، چون زمان‌بندی نویز برای داده‌ای با واریانس حدود ۱ تنظیم شده است. در شکل حرف K، نقاط تیز و محل اتصال خطوط سخت‌ترین بخش‌اند و معمولاً کمی تار بازسازی می‌شوند.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

### تمرین ۵

موضوع پروژه پایانی خود را انتخاب کنید و یک صفحه پیشنهاد شامل مسئله، داده، معیار ارزیابی و خط مبنا بنویسید.

</div>

<div dir="rtl" style="font-family: 'Vazirmatn FD', Vazirmatn, Tahoma, sans-serif;">

این تمرین کدنویسی ندارد. یک الگوی پیشنهادی برای صفحهٔ پیشنهاد پروژه، همراه با یک نمونهٔ پرشده:

| بخش | پرسش‌هایی که باید جواب دهید | نمونه: تشخیص بیماری برگ گوجه‌فرنگی |
|---|---|---|
| **مسئله** | چه چیزی را پیش‌بینی یا تولید می‌کنید؟ چه کسی از نتیجه استفاده می‌کند؟ | تشخیص ۱۰ نوع بیماری برگ گوجه‌فرنگی از روی عکس گوشی، برای کمک به کشاورزان |
| **داده** | منبع، تعداد نمونه، برچسب‌ها، مجوز استفاده، توازن کلاس‌ها | PlantVillage (بخش گوجه‌فرنگی)، حدود ۱۸ هزار تصویر، مجوز CC BY-SA |
| **تقسیم داده** | Train / Validation / Test؛ آیا خطر نشت داده وجود دارد؟ | ۷۰/۱۵/۱۵ به تفکیک کلاس؛ عکس‌های یک برگ فقط در یک بخش |
| **معیار ارزیابی** | چه معیاری و چرا؟ | Macro-F1 (کلاس‌ها نامتوازن‌اند) و دقت Top-1 |
| **خط مبنا** | ساده‌ترین روش معقول | Logistic Regression روی هیستوگرام رنگ؛ و ResNet-18 در حالت Feature Extraction |
| **روش اصلی** | مدل و دلیل انتخاب آن | Fine-tuning مدل EfficientNet-B0 با Augmentation (جلسهٔ ۹) |
| **ریسک‌ها** | چه چیزی ممکن است کار نکند؟ | عکس‌های آزمایشگاهی با پس‌زمینهٔ یکنواخت؛ عملکرد روی عکس‌های مزرعه ممکن است افت کند. چند عکس واقعی هم جمع می‌کنیم. |
| **تحویل** | خروجی نهایی | نوت‌بوک Colab، مدل روی Hugging Face Hub و README با جدول نتایج |

دو اشتباه رایج در پیشنهاد پروژه: انتخاب دادهٔ بسیار کوچک یا بدون برچسب، و نداشتن خط مبنا. بدون خط مبنا نمی‌توان گفت مدل عمیق واقعاً ارزش افزوده دارد یا نه. هر جای این جدول که مبهم است، باید پیش از نوشتن اولین خط کد روشن شود.

</div>